# Formação de Pares Própria — Cointegração Não Paramétrica

Até aqui o método NP operava os pares da cointegração **linear** (`pairs_formation_np.ipynb`, com
`ORIGEM_PARES` apontando para a formação linear). Este notebook seleciona os pares com critério
**próprio**: o par entra se houver equilíbrio ao longo da **curva**, testado com um valor crítico
calibrado para o nosso procedimento (`valor_critico_np.ipynb`). A negociação não muda: o arquivo de
saída tem as mesmas colunas e é lido por `pairs_trading_np.ipynb` via `ARQUIVO_PARES`.

### Passos, em cada janela de formação (12 meses, passo mensal)

| Passo | O que faz | Referência / motivo |
|---|---|---|
| 0 | Valor crítico do teste, simulado (notebook `valor_critico_np.ipynb`) | MacKinnon (1991, 2010): tabelas de cointegração são simulação; Phillips (2009): a curva "inventa" cointegração, a tabela de −3,34 não vale |
| 1 | Universo: ações com dados completos na janela, sem mesmo emissor | igual à formação linear (comparação pareada) |
| 2 | $Q = P/P_0$ | igual aos outros métodos |
| 3 | Pré-filtro: os `N_PRE_SSD` pares de menor SSD | Rad, Low & Faff (2016) § 4.2 (eles usam o SSD como pré-filtro da cointegração); N maior que os 100 da linear, para dar espaço a relações curvas |
| 4 | Curva nas **duas direções**; fica a de menor erro de CV relativo | a curva não é simétrica; hoje a direção era herdada da linear |
| 5 | **Teste**: ADF dos erros fora da amostra < valor crítico simulado | Engle & Granger (1987) em dois passos, com a curva no lugar da reta; erros fora da amostra para a curva não "copiar" o caminho |
| 6 | Filtro de hedge: mediana do β local ≥ 0,50 | coerente com o piso de β da negociação (DECISOES.md § 7) |
| 7 | Ranking → top 20 | padrão `'ssd'`: testa em ordem de SSD e para nos 20 primeiros aprovados, como a formação linear (Rad et al. 2016); alternativa `'adf'` |

Não usamos o teste de Wang & Phillips (2012) para selecionar: ele testa a **forma** da relação
(reta × curva) supondo que ela exista, e num par sem cointegração o resíduo linear passeia e o teste
rejeita "reta" com força — premiaria pares quebrados. Além disso, nos nossos dados o ganho da curva
sobre a reta na formação não tem relação com o resultado na negociação (correlação de postos −0,01;
DECISOES.md § 8).

In [ ]:
import json
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from formacao_utils import filtrar_ativos_completos, normalizar_precos, janelas_formacao, seleciona_pares_janela

warnings.filterwarnings('ignore')

# ── Janelas (iguais às dos outros métodos) ────────────────────────────────────
MESES_FORMACAO   = 12
MESES_NEGOCIACAO = 6
PASSO_MESES      = 1

# ── Seleção ───────────────────────────────────────────────────────────────────
N_PRE_SSD    = 300     # candidatos por janela (linear: 100)
TOP_K        = 20      # pares por janela (Rad et al. 2016)
SSD_MIN      = 0.01    # ghost tickers (igual à linear)
RANKING      = 'ssd'   # 'ssd': em ordem de SSD, para nos TOP_K aprovados (como a linear)
                       # 'adf': testa todos e fica com as TOP_K estatísticas mais negativas
BETA_MED_MIN = 0.50    # mediana do β local na formação (= piso de β da negociação)

# ── Teste (TEM de ser igual ao da simulação do valor crítico) ─────────────────
GRADE_H_REL      = np.r_[np.geomspace(0.05, 5, 25), np.inf]
GAP_UNILATERAL   = 20
T_MIN_UNILATERAL = 126
AUTOLAG_ADF      = 'AIC'
ARQ_CRITICO      = 'output/valor_critico/valor_critico_np.json'
NIVEL            = '5%'

# ── Execução e saída ──────────────────────────────────────────────────────────
N_PROCESSOS   = 3      # janelas em paralelo (1 = serial)
LIMITE_JANELAS = None  # só para testes: processa as primeiras N janelas
ARQUIVO_SAIDA = 'output/pares/pares_formados_np_propria.csv'
META_COLS = ['year', 'semester', 'period', 'day_in_semester', 'total_days_sem']

## 1. Valor crítico e conferência do procedimento

In [ ]:
crit_json = json.load(open(ARQ_CRITICO, encoding='utf-8'))
p_sim = crit_json['parametros']
grade_sim = [np.inf if v is None else v for v in p_sim['grade_h_rel']]
assert np.allclose(grade_sim, GRADE_H_REL) and p_sim['gap_unilateral'] == GAP_UNILATERAL \
    and p_sim['t_min_unilateral'] == T_MIN_UNILATERAL and p_sim['autolag_adf'] == AUTOLAG_ADF, \
    'O procedimento difere do usado na simulação: rode valor_critico_np.ipynb de novo.'
CRIT = crit_json['valor_critico'][NIVEL]
print(f"Valor crítico ({NIVEL}, nula '{crit_json['nula_principal']}', gerado em {crit_json['gerado_em']}): {CRIT:.3f}")
print(f"Para comparação: Engle–Granger 5% = −3,34; a tabela aprovaria "
      f"{crit_json['falso_positivo_com_EG_334']:.1%} dos pares SEM cointegração.")

## 2. Dados e janelas

In [ ]:
df = pd.read_csv('../../data_bases/base_unificada.csv', parse_dates=['date']).set_index('date')
colunas = [c for c in df.columns if c not in META_COLS]
meses_linha = pd.PeriodIndex(df.index, freq='M')
janelas = janelas_formacao(df.index, MESES_FORMACAO, MESES_NEGOCIACAO, PASSO_MESES)
if LIMITE_JANELAS:
    janelas = janelas[:LIMITE_JANELAS]
print(f"Janelas: {len(janelas)} ({janelas[0][0][0]} → {janelas[-1][1][-1]})")

## 3. Seleção em todas as janelas

A lógica de cada janela está em `formacao_utils.seleciona_pares_janela` (passos 3 a 7), para poder
rodar janelas em paralelo. Leva ~1–3 h com 3 processos.

In [ ]:
params = {'n_pre_ssd': N_PRE_SSD, 'ssd_min': SSD_MIN, 'grade_rel': GRADE_H_REL, 'gap': GAP_UNILATERAL,
          't_min': T_MIN_UNILATERAL, 'autolag': AUTOLAG_ADF, 'crit': CRIT, 'beta_med_min': BETA_MED_MIN,
          'top_k': TOP_K, 'ranking': RANKING}

tarefas = []
for mf, mn in janelas:
    Q = normalizar_precos(filtrar_ativos_completos(df.loc[meses_linha.isin(mf), colunas]))
    tarefas.append(({'janela': f"{mf[0]}_a_{mf[-1]}", 'formacao_inicio': str(mf[0]), 'formacao_fim': str(mf[-1]),
                     'negociacao_inicio': str(mn[0]), 'negociacao_fim': str(mn[-1]), 'n_ativos': Q.shape[1]}, Q))

t0 = time.time()
resultados, logs = [], []
if N_PROCESSOS > 1:
    with ProcessPoolExecutor(N_PROCESSOS) as ex:
        saidas = ex.map(seleciona_pares_janela, [Q for _, Q in tarefas], [params] * len(tarefas))
        for k, ((info, _), (sel, log)) in enumerate(zip(tarefas, saidas)):
            resultados += [{**info, 'rank': r + 1, **par} for r, par in enumerate(sel)]
            logs.append({**info, **log})
            if (k + 1) % 25 == 0:
                print(f"  {k + 1:3d} / {len(tarefas)} janelas  ({time.time() - t0:5.0f} s)", flush=True)
else:
    for k, (info, Q) in enumerate(tarefas):
        sel, log = seleciona_pares_janela(Q, params)
        resultados += [{**info, 'rank': r + 1, **par} for r, par in enumerate(sel)]
        logs.append({**info, **log})
        if (k + 1) % 25 == 0:
            print(f"  {k + 1:3d} / {len(tarefas)} janelas  ({time.time() - t0:5.0f} s)", flush=True)
print(f"Tempo: {time.time() - t0:.0f} s")

## 4. Salvando no formato da negociação

As colunas seguem `pares_formados_np.csv`, para `pairs_trading_np.ipynb` ler sem mudanças:
`spread_media` e `spread_desvio` são a média e o desvio dos erros um passo à frente (o mesmo
`SIGMA_MODO = 'unilateral'` da formação com pares lineares); `adf_np` é a estatística do teste.

In [ ]:
pares = pd.DataFrame(resultados).rename(columns={'mu': 'spread_media', 'sigma': 'spread_desvio', 'stat': 'adf_np',
                                                 'cv_rel': 'cv_mse', 'cv_rel_linear': 'cv_mse_linear'})
pares['origem'] = 'propria'
pares['criterio_h'] = f'unilateral_g{GAP_UNILATERAL}'
pares['sigma_modo'] = 'unilateral'
pares['valor_critico'] = CRIT
log_df = pd.DataFrame(logs)
pares.to_csv(ARQUIVO_SAIDA, index=False)
log_df.to_csv(ARQUIVO_SAIDA.replace('.csv', '_log.csv'), index=False)
print(f"Pares: {len(pares):,} em {pares['janela'].nunique()} janelas → {ARQUIVO_SAIDA}")

## 5. Diagnóstico

In [ ]:
print(f"Pares por janela: média {log_df['n_selecionados'].mean():.1f} | janelas com {TOP_K}: "
      f"{(log_df['n_selecionados'] >= TOP_K).mean():.1%} | janelas vazias: {(log_df['n_selecionados'] == 0).sum()}")
taxa = log_df['n_coint'].sum() / log_df['n_testados'].sum()
print(f"Candidatos testados: {log_df['n_testados'].sum():,} | aprovados no teste: {taxa:.1%} "
      f"| aprovados também no filtro de β: {log_df['n_aprovados'].sum() / max(log_df['n_coint'].sum(), 1):.1%}")
print(f"Reta escolhida (h = ∞) entre os selecionados: {pares['h_linear'].mean():.1%}")
print(f"Estatística ADF dos selecionados: mediana {pares['adf_np'].median():.2f} (valor crítico {CRIT:.2f})")

# Sobreposição com os pares da formação linear (mesma janela, mesmo par sem ordem)
lin = pd.read_csv('../cointegration/output/pares/pares_formados_coint.csv')
chave = lambda d: d['janela'] + '|' + d[['ativo1', 'ativo2']].apply(lambda r: '|'.join(sorted(r)), axis=1)
comum = set(chave(pares)) & set(chave(lin))
print(f"Pares também escolhidos pela formação linear: {len(comum) / len(pares):.1%} "
      f"({len(comum):,} de {len(pares):,})")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
log_df.set_index(pd.PeriodIndex(log_df['negociacao_inicio'], freq='M').to_timestamp())['n_selecionados'].plot(ax=axes[0], color='#E65100')
axes[0].set_title('Pares selecionados por janela', fontsize=10)
axes[0].axhline(TOP_K, color='gray', ls='--', lw=0.8)
axes[0].grid(alpha=0.3)
axes[1].hist(pares['adf_np'], bins=40, color='#E65100', alpha=0.7)
axes[1].axvline(CRIT, color='black', lw=1.2, label=f'valor crítico {CRIT:.2f}')
axes[1].set_title('Estatística do teste nos pares selecionados', fontsize=10)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/imagens/formacao_propria_diagnostico.png', dpi=150, bbox_inches='tight')
plt.show()

### Referências

- Rad, H., Low, R. & Faff, R. (2016). *The profitability of pairs trading strategies: distance, cointegration and copula methods.* Quantitative Finance 16(10) — pré-filtro SSD seguido de teste de cointegração; 20 pares por janela.
- Engle, R. & Granger, C. (1987). Econometrica 55(2) — teste de cointegração em dois passos baseado no resíduo.
- MacKinnon, J. (1991, 2010) — valores críticos de cointegração obtidos por simulação.
- Phillips, P. (2009). Econometric Theory 25(6) — regressão não paramétrica espúria.
- Wang, Q. & Phillips, P. (2012). Annals of Statistics 40(2) — teste de especificação (não usado na seleção; ver introdução).
- Wang, Q. & Phillips, P. (2023). Econometric Theory 39(6) — bandwidth ótimo com regressor I(1) (fundamento da escolha de h por CV).
- Slides *Cointegração Não Paramétrica via Kernel* (ECO 02091, UFRGS), 21–24 e 33.